# Chapter 3 — Gender and Market Segmentation

Gender-specific catalogue and accord-combination analysis across Men, Women and Unisex classifications. Median segmentation is calculated independently within each classification and analytical level.

Votes indicate platform activity, not sales, market share or unique consumers. Ratings are community evaluations. Catalogue classifications are not evidence of innate preferences.

## 3.1 Configuration and source validation

Raw Fragrantica data supplies ratings, vote counts, classification and ordered accords. Notebook 13's prepared table supplies separately cleaned fragrance and brand names. The join uses URL-derived fragrance_id.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from pathlib import Path

SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
REFERENCE_TABLE = "main.fragrance_analytics.fragrantica_prepared"
OUTPUT_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/chapter_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDER_ORDER = ["Men", "Women", "Unisex"]
SEGMENT_ORDER = ["Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"]
MIN_DOMINANT_FRAGRANCES = 100
MIN_DYAD_FRAGRANCES = 100
MIN_TRIAD_FRAGRANCES = 50

df_source = spark.table(SOURCE_TABLE)
df_reference = spark.table(REFERENCE_TABLE)
print("Source rows:", f"{df_source.count():,}")
print("Prepared reference rows:", f"{df_reference.count():,}")
df_source.printSchema()
df_reference.printSchema()

df_source_keyed = df_source.withColumn(
    "fragrance_id",
    F.regexp_extract("url", r"-([0-9]+)\\.html(?:\\?.*)?$", 1)
)
assert df_reference.groupBy("fragrance_id").count().filter(F.col("count") > 1).limit(1).count() == 0, "Duplicate reference fragrance_id values."

df_names = df_reference.select(
    "fragrance_id",
    F.col("fragrance_name_clean").alias("fragrance_name"),
    F.col("brand_clean").alias("brand")
)
df_joined = df_source_keyed.join(df_names, "fragrance_id", "left")
source_rows = df_source_keyed.count()
joined_rows = df_joined.count()
assert joined_rows == source_rows, f"Reference join changed row count: {source_rows} -> {joined_rows}"
print("Join rows preserved:", f"{joined_rows:,}")
print("Missing cleaned names:", df_joined.filter(F.col("fragrance_name").isNull() | (F.trim("fragrance_name") == "")).count())
print("Missing cleaned brands:", df_joined.filter(F.col("brand").isNull() | (F.trim("brand") == "")).count())

df_accords = (
    df_joined
    .withColumn("accord_array", F.from_json(
        F.regexp_replace(F.regexp_replace(F.col("main_accords"), "'", '"'), r"\\bNone\\b", "null"),
        "array<string>"
    ))
    .withColumn("gender_clean", F.initcap(F.lower(F.trim("gender"))))
    .filter(F.col("gender_clean").isin(GENDER_ORDER))
    .withColumn("accord_array", F.expr("filter(transform(accord_array, x -> lower(trim(x))), x -> x is not null and x <> '')"))
    .filter(F.col("id").isNotNull() & (F.size("accord_array") > 0))
    .dropDuplicates(["id"])
)
assert df_accords.groupBy("id").count().filter(F.col("count") > 1).limit(1).count() == 0
print("Classified fragrances with valid accords:", f"{df_accords.count():,}")
display(df_accords.groupBy("gender_clean").agg(F.countDistinct("id").alias("fragrance_count")).orderBy("gender_clean"))